# Strategy Comparison: FedGRA vs. High-Loss

Compare client selection strategies on MNIST:
- **FedGRA** (baseline, `min_participate_round=10`)
- **High-Loss** (selects clients with highest loss)

In [1]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add root for style
root_dir = Path("../../..").resolve()
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
from style import MatplotlibStyle
MatplotlibStyle().apply()

print("✅ Imports ready.")


✅ Imports ready.


In [2]:
# ============================================================
# Load both CSV files
# ============================================================

FILES = {
    "FedGRA":    "fedgra_keras_exact_mnist_-train-20260615_023523-c2a79bf0.csv",
    "High-Loss": "fedgra_keras_exact_high_loss_mnist_-train-20260615_042615-c17bfe7d.csv",
}

def extract_method_name(csv_path: str) -> str:
    """Extract client_selection.method from first-line JSON config."""
    with open(csv_path, "r", encoding="utf-8") as f:
        first_line = f.readline().strip()
    if first_line.startswith("Config,"):
        cfg = json.loads(first_line[len("Config,"):])
        return cfg.get("client_selection", {}).get("method", "unknown")
    return "unknown"

def load_csv(csv_path: str) -> pd.DataFrame:
    """Skip Config line + blank line, header on row 2."""
    df = pd.read_csv(csv_path, skiprows=[0, 1])
    if "round" in df.columns:
        df = df.set_index("round")
    else:
        df.index = range(1, len(df) + 1)
    return df

data_dict = {}
for label, filename in FILES.items():
    filepath = Path(filename)
    if not filepath.exists():
        print(f"⚠️  Missing: {filename}")
        continue
    method = extract_method_name(str(filepath))
    df = load_csv(str(filepath))
    data_dict[label] = df
    print(f"✅ {label:12s}  method={method:12s}  rounds={len(df)}  "
          f"best_acc={df['accuracy'].max():.4f} (r{df['accuracy'].idxmax()})  "
          f"final_acc={df['accuracy'].iloc[-1]:.4f}")


✅ FedGRA        method=fedgra        rounds=101  best_acc=0.9367 (r94)  final_acc=0.9297
✅ High-Loss     method=high_loss     rounds=101  best_acc=0.8873 (r98)  final_acc=0.8834


In [ ]:
# ============================================================
# Prepare smoothed data
# ============================================================

WINDOW = 5

def rolling_mean(series, window):
    return series.rolling(window=window, min_periods=1).mean()

plot_data = {}
for name, df in data_dict.items():
    plot_data[name] = {
        "rounds":         df.index.values,
        "acc_raw":        df["accuracy"].values,
        "loss_raw":       df["average_loss"].values,
        "acc_smooth":     rolling_mean(df["accuracy"], WINDOW).values,
        "loss_smooth":    rolling_mean(df["average_loss"], WINDOW).values,
    }


In [ ]:
# ============================================================
# Plot: Accuracy + Loss (side-by-side)
# ============================================================

COLORS = {"FedGRA": "#2a7de1", "High-Loss": "#e53e3e"}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Accuracy ---
ax = axes[0]
for name, data in plot_data.items():
    c = COLORS[name]
    # Raw
    ax.plot(data["rounds"], data["acc_raw"], color=c, linewidth=0.8, alpha=0.4)
    # Smoothed
    ax.plot(data["rounds"], data["acc_smooth"], color=c, linewidth=2.2, label=name)
    # Max marker
    idx = np.argmax(data["acc_raw"])
    ax.scatter(data["rounds"][idx], data["acc_raw"][idx],
               color=c, marker='*', s=200, zorder=5, edgecolors='black', linewidths=0.5)

ax.set_title("Test Accuracy", fontsize=22)
ax.set_xlabel("Communication Round", fontsize=20)
ax.set_ylabel("Accuracy", fontsize=20)
ax.set_ylim(0, 1.0)
ax.legend(fontsize=15, loc="lower right")
ax.grid(True, alpha=0.3)

# --- Loss ---
ax = axes[1]
for name, data in plot_data.items():
    c = COLORS[name]
    ax.plot(data["rounds"], data["loss_raw"], color=c, linewidth=0.8, alpha=0.4)
    ax.plot(data["rounds"], data["loss_smooth"], color=c, linewidth=2.2, label=name)

ax.set_title("Average Loss", fontsize=22)
ax.set_xlabel("Communication Round", fontsize=20)
ax.set_ylabel("Loss", fontsize=20)
ax.legend(fontsize=15, loc="upper right")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
